In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
from pipeline_core import (
    prepare_preds_for_peaks_onlyRC,
    peak_finding,
    find_tss_polya_pairs_right_left_only,
    filter_bed_by_intragenic,
)

# ============================ CONFIG ============================
chrom = 'NC_060944.1'
chrom_len = 66210255
files_path = '/disk/10tb/home/fishman/DNALM/GENA_LM/runs/annotation/modernGENAlarge_ep36_rc_shift/checkpoint-38750/eval/T2T-CHM13v2/NC_060944.1/'

LP_FRAC   = 0.05
PK_PROM   = 0.1
PK_DIST   = 50
PK_HEIGHT = None

args_prob_threshold = float(0.5)
args_zero_fraction_drop_threshold = float(0.01)
args_bed_out = f"../../data/tmp/moderngena_shawerma_thr_{PK_PROM}.bed"

BW_PLUS          = "intragenic_+.bw"
BW_MINUS         = "intragenic_-.bw"
BW_PLUS_RC       = "intragenic_+rev_comp_.bw"
BW_MINUS_RC      = "intragenic_-rev_comp_.bw"

args_bw_dir = '/mnt/nfs_dna/minja/tmp/MLGENX_modernGENA_rc_shift_middle_pretrain_shawerma_6_classes_1024/eval/T2T-CHM13v2/NC_060944.1'

if not os.path.exists(os.path.dirname(args_bed_out)):
    os.makedirs(os.path.dirname(args_bed_out))
# ================================================================


# ----------------START OF THE PIPELINE----------------
# this is four bigWig output
tss_plus, polya_plus, tss_minus, polya_minus = prepare_preds_for_peaks_onlyRC(files_path)

X = np.array([
	tss_plus,
	polya_plus,
	tss_minus,
	polya_minus
])

arr = peak_finding(X, LP_FRAC=LP_FRAC, PK_PROM=PK_PROM, PK_DIST=PK_DIST, PK_HEIGHT=PK_HEIGHT)

pairs = find_tss_polya_pairs_right_left_only(arr, k=10, chrom_name=chrom, out_bed_path=None, progress_every=1000)
print(f"Wrote {len(pairs)} intervals to file.")

filter_bed_by_intragenic(pairs, args_bw_dir, BW_PLUS, BW_MINUS, BW_PLUS_RC, BW_MINUS_RC,
						args_prob_threshold,
						args_zero_fraction_drop_threshold, args_bed_out)


Reading tss_+.bw: chromosome NC_060944.1 (length: 66210255)
Reading tss_+rev_comp_.bw: chromosome NC_060944.1 (length: 66210255)
Reading tss_-.bw: chromosome NC_060944.1 (length: 66210255)
Reading tss_-rev_comp_.bw: chromosome NC_060944.1 (length: 66210255)
Reading polya_+.bw: chromosome NC_060944.1 (length: 66210255)
Reading polya_+rev_comp_.bw: chromosome NC_060944.1 (length: 66210255)
Reading polya_-.bw: chromosome NC_060944.1 (length: 66210255)
Reading polya_-rev_comp_.bw: chromosome NC_060944.1 (length: 66210255)
tss_+.max = 0.6893149614334106
tss_+rev_comp_.max = 0.7340891361236572
tss_-.max = 0.7483487725257874
tss_-rev_comp_.max = 0.6799196004867554
polya_+.max = 0.9388062953948975
polya_+rev_comp_.max = 0.9112286567687988
polya_-.max = 0.8860156536102295
polya_-rev_comp_.max = 0.8607997894287109
Input shape: (4, 662102)
Scanning 10 plus TSS seeds one sided
Scanning 8 minus TSS seeds one sided
Wrote 110 intervals to file.
[INFO] Mode: STRANDED (4 bigWigs in directory)
  - intra

Filtering: 100%|██████████| 110/110 [00:04<00:00, 24.07interval/s]

[DONE] Kept 24 / 110 intervals (21.82%). Dropped 86 (78.18%).
[INFO] Intervals left after filtration: 24


In [5]:
import pysam
import tempfile
with tempfile.NamedTemporaryFile(delete=False) as tmp_file:
	tmp_file.write(">test\nATGCATGC".encode())
	print(tmp_file.name)

/tmp/tmp6y715e99


>test
ATGCATGC